# 12.07 - Transfer Learning Theory

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Transfer-learning skeleton coded from memory.

Today is about using a pretrained-style backbone without turning the notebook into a giant training run. You will replace a classifier head, freeze and unfreeze feature-extractor layers, run one small training epoch, and inspect whether the model wiring is correct.


## Core Ideas

Transfer learning starts from a backbone that already knows useful visual features, then adapts the final layers to a new label set.

- **Backbone:** the feature extractor, often a ResNet, EfficientNet, ConvNeXt, or ViT. It turns an image tensor into a compact representation.
- **Classifier head:** the final task-specific layer. For a new contest dataset, the head output size must equal the number of classes.
- **Freezing:** setting backbone parameters to `requires_grad=False` so early training only updates the new head.
- **Unfreezing:** later allowing some or all backbone parameters to train, usually with a smaller learning rate.
- **Torchvision models:** `torchvision` is an allowed image library in `RESOURCE/library.png`; this notebook uses `torchvision.models.resnet18` directly. The code uses `weights=None` to avoid an internet download, but the same head-replacement pattern applies to cached pretrained weights.
- **Validation transforms:** should stay deterministic so metric changes come from the model, not random augmentation.

The practical goal is not high accuracy on the toy dataset. The goal is a reliable skeleton you can write from memory under contest pressure.


In [1]:
import csv
import os

import numpy as np
from PIL import Image, ImageDraw, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as tv_models
import torchvision.transforms as T

SEED = 42
DATA_DIR = "_day12_image_data"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
LABELS_CSV = os.path.join(DATA_DIR, "labels.csv")
CLASSES = ("circle", "square", "triangle")
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("torch:", torch.__version__, "torchvision models/transforms: available")


torch: 2.13.0+cpu torchvision models/transforms: available


## Prepared Image Data

Run this cell before the exercises. It writes a tiny three-class image dataset into `_day12_image_data/` with `train` and `val` splits. The dataset is intentionally simple so the transfer-learning code can stay fast and self-contained.


In [2]:
def make_transfer_demo_dataset(out_dir=DATA_DIR, image_size=64, n_per_class=10, seed=42):
    rng = np.random.default_rng(seed)
    image_dir = os.path.join(out_dir, "images")
    os.makedirs(image_dir, exist_ok=True)

    records = []
    base_colors = {
        "circle": (225, 75, 90),
        "square": (70, 170, 95),
        "triangle": (65, 120, 225),
    }

    for label in CLASSES:
        for idx in range(n_per_class):
            bg = rng.integers(18, 48, size=(image_size, image_size, 3), dtype=np.uint8)
            img = Image.fromarray(bg)
            draw = ImageDraw.Draw(img)
            color = tuple(
                min(255, max(0, channel + int(rng.integers(-18, 19))))
                for channel in base_colors[label]
            )
            margin = int(rng.integers(11, 17))
            jitter = int(rng.integers(-3, 4))
            box = [margin + jitter, margin, image_size - margin, image_size - margin - jitter]

            if label == "circle":
                draw.ellipse(box, fill=color)
                draw.ellipse([24, 24, 40, 40], fill=(245, 235, 105))
            elif label == "square":
                draw.rectangle(box, fill=color)
                draw.line([14, image_size // 2, image_size - 14, image_size // 2], fill=(245, 235, 105), width=3)
            else:
                points = [
                    (image_size // 2, margin - 2),
                    (image_size - margin + jitter, image_size - margin),
                    (margin - jitter, image_size - margin),
                ]
                draw.polygon(points, fill=color)
                draw.line(points + [points[0]], fill=(245, 235, 105), width=2)

            if idx % 2 == 1:
                img = ImageEnhance.Contrast(img).enhance(1.08)
            if idx % 3 == 0:
                img = ImageEnhance.Brightness(img).enhance(1.06)

            noise = rng.normal(0, 5, size=(image_size, image_size, 3))
            arr = np.clip(np.asarray(img, dtype=np.float32) + noise, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)

            filename = f"{label}_{idx:02d}.png"
            rel_path = os.path.join("images", filename).replace(os.sep, "/")
            split = "val" if idx >= n_per_class - 2 else "train"
            img.save(os.path.join(image_dir, filename))
            records.append({"image_path": rel_path, "label": label, "split": split})

    with open(os.path.join(out_dir, "labels.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label", "split"])
        writer.writeheader()
        writer.writerows(records)

    return records


records = make_transfer_demo_dataset()
print(f"Wrote {len(records)} images to {IMAGE_DIR}")


Wrote 30 images to _day12_image_data\images


## Helper Functions

These helpers keep the exercises focused on transfer learning while using only libraries allowed by `RESOURCE/library.png`: `torch`, `torchvision`, `numpy`, `csv`, `os`, and `Pillow`.


In [3]:
def load_rgb_image(path):
    return Image.open(path).convert("RGB")


def split_records(records):
    train_records = [record for record in records if record["split"] == "train"]
    val_records = [record for record in records if record["split"] == "val"]
    return train_records, val_records


def count_parameters(model):
    total = sum(param.numel() for param in model.parameters())
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    return {"total": total, "trainable": trainable, "frozen": total - trainable}


def classifier_parameters(model):
    return list(model.fc.parameters())


## Exercise 12-A: Records and Label Mapping

Read `labels.csv`, turn relative image paths into full `Path` objects, verify that every file exists, and build stable label maps. Stable maps matter because the model output index must always mean the same class.


In [ ]:
# TODO 12-A
import pandas as pd
def load_transfer_records(labels_csv=LABELS_CSV, data_dir=DATA_DIR):
    df = pd.read_csv(labels_csv)
    for idx, rows in df.iterrows() : 
        path = rows["image_path"]
        
    raise NotImplementedError("Read labels.csv and return records with image_path, label, and split fields.")


def build_label_maps(records):
    raise NotImplementedError("Return (label_to_idx, idx_to_label) using sorted unique labels.")


records = load_transfer_records()
label_to_idx, idx_to_label = build_label_maps(records)
print(label_to_idx)


## Exercise 12-B: Transforms, Dataset, and DataLoaders

Build deterministic validation preprocessing and a light training transform with `torchvision.transforms`. Then wrap the records in a `Dataset` that returns `(image_tensor, label_index)` and create train/validation loaders.


In [ ]:
# TODO 12-B

class TransferImageDataset(Dataset):
    def __init__(self, records, label_to_idx, transform=None):
        raise NotImplementedError("Store records, label map, and transform.")

    def __len__(self):
        raise NotImplementedError("Return the number of records.")

    def __getitem__(self, index):
        raise NotImplementedError("Return (image_tensor, label_index).")


def build_transfer_transforms(image_size=64, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    raise NotImplementedError("Return torchvision train and validation transform pipelines.")


def create_transfer_loaders(records, label_to_idx, batch_size=8, image_size=64, seed=42):
    raise NotImplementedError("Return (train_loader, val_loader) for train and validation records.")


## Exercise 12-C: Replace the Classifier Head

Create a transfer-learning model with `torchvision.models.resnet18`, replace its `fc` layer for the new number of classes, and freeze the backbone when requested.


In [ ]:
# TODO 12-C

def make_transfer_model(num_classes, freeze_backbone=True):
    raise NotImplementedError("Return (model, info) after replacing the torchvision classifier head.")


def set_backbone_trainable(model, trainable):
    raise NotImplementedError("Toggle backbone parameters and keep classifier-head parameters trainable.")


def count_trainable_parameters(model):
    raise NotImplementedError("Return the number of parameters with requires_grad=True.")


## Exercise 12-D: One Training Epoch

Train only the parameters that require gradients, then evaluate on validation data. Keep the returned metrics small and explicit: loss, accuracy, and number of examples.


In [ ]:
# TODO 12-D

def train_one_epoch(model, loader, criterion, optimizer, device):
    raise NotImplementedError("Run one training epoch and return metrics.")


def evaluate_classifier(model, loader, criterion, device):
    raise NotImplementedError("Evaluate without gradients and return metrics.")


## Exercise 12-E: Transfer-Learning Summary

Create a concise dictionary that records the model source, head type, class names, frozen/trainable parameter counts, and train/validation metrics. This is the kind of evidence you want after a quick contest experiment.


In [ ]:
# TODO 12-E

def summarize_transfer_setup(model, model_info, label_to_idx, train_metrics=None, val_metrics=None):
    raise NotImplementedError("Return a compact dictionary describing the transfer-learning run.")


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 12 tests passed`.


In [ ]:
def run_day12_tests():
    required_names = [
        "load_transfer_records",
        "build_label_maps",
        "TransferImageDataset",
        "build_transfer_transforms",
        "create_transfer_loaders",
        "make_transfer_model",
        "set_backbone_trainable",
        "count_trainable_parameters",
        "train_one_epoch",
        "evaluate_classifier",
        "summarize_transfer_setup",
    ]
    for name in required_names:
        assert name in globals(), f"Missing required name: {name}"

    assert os.path.exists(LABELS_CSV), "labels.csv was not created"
    assert os.path.isdir(IMAGE_DIR), "image directory was not created"
    image_files = sorted(name for name in os.listdir(IMAGE_DIR) if name.endswith(".png"))
    assert len(image_files) == 30, "Expected 30 generated images"

    records = load_transfer_records()
    assert len(records) == 30, len(records)
    assert all(os.path.exists(record["image_path"]) for record in records), "Every image path must exist"
    assert sorted({record["label"] for record in records}) == ["circle", "square", "triangle"]
    assert sorted({record["split"] for record in records}) == ["train", "val"]

    label_to_idx, idx_to_label = build_label_maps(records)
    assert label_to_idx == {"circle": 0, "square": 1, "triangle": 2}, label_to_idx
    assert idx_to_label[2] == "triangle"

    train_transform, val_transform = build_transfer_transforms(image_size=64)
    sample_tensor = val_transform(load_rgb_image(records[0]["image_path"]))
    assert sample_tensor.shape == (3, 64, 64), sample_tensor.shape
    assert sample_tensor.dtype == torch.float32
    assert torch.isfinite(sample_tensor).all()

    dataset = TransferImageDataset(records[:4], label_to_idx, transform=val_transform)
    image_tensor, label_idx = dataset[0]
    assert image_tensor.shape == (3, 64, 64)
    assert isinstance(label_idx, int)
    assert 0 <= label_idx < 3

    train_loader, val_loader = create_transfer_loaders(records, label_to_idx, batch_size=6, image_size=64, seed=123)
    images, targets = next(iter(train_loader))
    assert images.shape == (6, 3, 64, 64), images.shape
    assert targets.dtype == torch.long
    assert targets.min().item() >= 0 and targets.max().item() < 3

    model, model_info = make_transfer_model(num_classes=3, freeze_backbone=True)
    assert model_info["source"] == "torchvision_resnet18"
    assert model_info["num_classes"] == 3
    assert model_info["backbone_frozen"] is True
    logits = model(images[:2])
    assert logits.shape == (2, 3), logits.shape

    counts = count_parameters(model)
    trainable_before = count_trainable_parameters(model)
    assert trainable_before == counts["trainable"]
    assert 0 < trainable_before < counts["total"], counts
    assert all(param.requires_grad for param in classifier_parameters(model)), "Classifier head must stay trainable"

    set_backbone_trainable(model, True)
    trainable_after = count_trainable_parameters(model)
    assert trainable_after > trainable_before
    set_backbone_trainable(model, False)
    assert count_trainable_parameters(model) == trainable_before

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, model_info = make_transfer_model(num_classes=3, freeze_backbone=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-3)
    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_classifier(model, val_loader, criterion, device)
    for metrics in (train_metrics, val_metrics):
        assert set(metrics) == {"loss", "accuracy", "n_examples"}, metrics
        assert metrics["loss"] >= 0 and np.isfinite(metrics["loss"])
        assert 0.0 <= metrics["accuracy"] <= 1.0
        assert metrics["n_examples"] > 0

    summary = summarize_transfer_setup(model, model_info, label_to_idx, train_metrics, val_metrics)
    assert summary["classes"] == ["circle", "square", "triangle"]
    assert summary["parameter_counts"]["trainable"] == count_trainable_parameters(model)
    assert "train" in summary and "val" in summary

    print("Day 12 tests passed")


run_day12_tests()


## Day 12 Checklist

Before moving on, make sure you can explain which module is the backbone, which module is the classifier head, why the head output size equals the class count, which parameters are frozen, which parameters the optimizer sees, and what changed after one training epoch.
